# Customer Retention — Exploratory Analysis

This notebook investigates the structure, quality, and behavioral patterns of the customer dataset before predictive modeling.

The analysis is intentionally exploratory: conclusions should be based on the outputs produced from the user's local dataset rather than assumptions made in advance.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

TRAIN_PATH = ROOT / "data" / "raw" / "train.csv"
TARGET = "Churn"

df = pd.read_csv(TRAIN_PATH)
df.head()

In [ ]:
print("Rows:", len(df))
print("Columns:", len(df.columns))

display(df.dtypes.to_frame("dtype"))
display(df.isna().sum().sort_values(ascending=False).to_frame("missing"))

In [ ]:
target_view = (
    df[TARGET]
    .astype(str)
    .str.strip()
    .str.lower()
    .value_counts(dropna=False)
    .rename_axis(TARGET)
    .reset_index(name="customers")
)

target_view["share"] = target_view["customers"] / len(df)
display(target_view)

In [ ]:
numeric = df.select_dtypes(include=np.number)

if not numeric.empty:
    display(numeric.describe().T)
else:
    print("No numeric columns detected.")

In [ ]:
categorical = df.select_dtypes(exclude=np.number).columns.tolist()

for column in categorical[:8]:
    print(f"\n--- {column} ---")
    display(df[column].value_counts(dropna=False).head(12).to_frame("count"))

In [ ]:
plot_candidates = [
    column for column in ["tenure", "MonthlyCharges", "TotalCharges"]
    if column in df.columns
]

for column in plot_candidates:
    values = pd.to_numeric(df[column], errors="coerce")
    plt.figure(figsize=(9, 4))
    sns.histplot(values.dropna(), bins=35, kde=True)
    plt.title(f"Distribution of {column}")
    plt.xlabel(column)
    plt.tight_layout()
    plt.show()

In [ ]:
if "tenure" in df.columns:
    analysis = df.copy()
    analysis["tenure_numeric"] = pd.to_numeric(
        analysis["tenure"], errors="coerce"
    )

    churn_text = (
        analysis[TARGET]
        .astype(str)
        .str.strip()
        .str.lower()
    )
    analysis["churn_flag"] = churn_text.map({"yes": 1, "no": 0, "1": 1, "0": 0})

    grouped = (
        analysis.groupby(pd.cut(analysis["tenure_numeric"], bins=6))[
            "churn_flag"
        ]
        .mean()
        .reset_index()
    )

    grouped.columns = ["tenure_band", "churn_rate"]
    display(grouped)

## Questions to answer from the outputs

- Which variables show the largest differences between retained and churned customers?
- Are there suspicious missing or malformed values that require treatment?
- Does tenure appear to have a nonlinear relationship with churn?
- Do contract categories behave differently?
- Does service adoption appear associated with retention?
- Which observations should influence the feature-engineering stage?

These questions should be answered using the actual charts and tables generated during the run.